<a href="https://colab.research.google.com/github/Abhi-pan1982/ML_NLP/blob/main/pdf_to_vector_storage.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install pypdf langchain openai faiss-cpu
import os
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.chains import RetrievalQA
from PyPDF2 import PdfReader

# ✅ Set your API key (better to use environment variable)
os.environ["OPENAI_API_KEY"] = "your_api_key_here"

# Step 1: Load PDF
def load_pdf(file_path):
    reader = PdfReader(file_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text() + "\n"
    return text

pdf_text = load_pdf("bank_policy_2002_2003.pdf")

# Step 2: Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_text(pdf_text)

# Step 3: Create embeddings + vector store
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(chunks, embeddings)

# Step 4: Build RAG pipeline
retriever = vectorstore.as_retriever()
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model="gpt-5.5-pro"),
    retriever=retriever,
    chain_type="stuff"
)

# Step 5: Ask questions
query = "What were the major lending policy changes in 2002–2003?"
answer = qa_chain.run(query)

print("Answer:", answer)
